## Zero-Shot LID Benchmark — ConLID

Adapted from `notebooks/FastText_Zeroshot.ipynb` / `notebooks/ConLID_zeroshot.ipynb`. Evaluates the off-the-shelf **ConLID** model, with no fine-tuning, on FLORES+ devtest sentences for our core languages (`sin`, `san`) plus confusable neighbours the model has to tell them apart from. `pli` (Pali) is excluded — FLORES+ has no Pali split.

Uses the pipeline's already-preprocessed `datasets/preprocessed/flores_plus.jsonl` (produced by `02.preprocess/preprocess_flores_plus.ipynb`) instead of re-downloading from Hugging Face.

**Caveat:** FLORES+ ships `arb` (Arabic) in both Arabic-script and romanized Latin-script variants, and `preprocess_flores_plus.ipynb` doesn't retain script metadata, so `arb` rows here are an unlabeled mix of both scripts (2x the row count of every other language). This doesn't affect the `sin`/`san` results this benchmark exists for, but treat Arabic's numbers as noisy.

In [ ]:
input_dir = 'datasets/preprocessed'
output_file = 'datasets/benchmark_results/conlid.csv'

In [ ]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import os

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

# FLORES+ iso_639_3 code -> FLORES-style script-tagged code (used by every model
# here except fastText LID-176, which speaks plain ISO 639-1).
TARGET_LANGUAGES = {
    "eng": "eng_Latn",  # English (baseline)
    "sin": "sin_Sinh",  # Sinhala
    "san": "san_Deva",  # Sanskrit
    "tam": "tam_Taml",  # Tamil
    "hin": "hin_Deva",  # Hindi
    "ben": "ben_Beng",  # Bengali
    "arb": "arb_Arab",  # Arabic (Modern Standard)
    "fra": "fra_Latn",  # French
    "deu": "deu_Latn",  # German
}

import glob

print(f"Loading datasets from {input_dir}...")
records = []
dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
for file_path in dataset_files:
    print(f"  Reading {os.path.basename(file_path)}...")
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)

df = pd.DataFrame(records)
df["flores_label"] = df["label"].map(TARGET_LANGUAGES)
print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")


def evaluate_and_save(results, model_name, target_labels):
    """results must have 'true_label' and 'predicted_label' columns."""
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    results.to_csv(output_file, index=False)
    print(f"\nSaved predictions to {output_file}")
    return results

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "conlid_repo"
REPO_URL = "https://github.com/epfl-nlp/language-identification.git"

if not os.path.exists(REPO_DIR):
    print("Cloning ConLID repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing ConLID requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)

sys.path.append(REPO_DIR)
from model import ConLID  # noqa: E402  (comes from the cloned repo)

from huggingface_hub import snapshot_download  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402

print("Downloading ConLID checkpoints...")
checkpoint_dir = os.path.join(REPO_DIR, "checkpoints", "conlid")
snapshot_download(repo_id="epfl-nlp/ConLID", local_dir=checkpoint_dir)

print("Loading ConLID model...")
conlid_model = ConLID.from_pretrained(dir=checkpoint_dir)

texts = df["text"].astype(str).tolist()

print(f"Evaluating {len(texts)} samples with ConLID...")
predicted_labels = []
for text in tqdm(texts):
    # predict() returns a tuple like: (['eng_Latn'], [0.97])
    pred_result = conlid_model.predict(text, k=1)
    predicted_labels.append(pred_result[0][0])

results = df[["text", "label", "source"]].copy()
results["true_label"] = df["flores_label"]
results["predicted_label"] = predicted_labels

target_labels = sorted(set(TARGET_LANGUAGES.values()))
results = evaluate_and_save(results, "ConLID", target_labels)
results